In [ ]:
!pip install scikit-learn

In [1]:
import os
import pandas as pd
import numpy as np
import joblib

# Fix for NumPy 2.0+ removal of np.long attribute
if not hasattr(np, "long"):
    np.long = int

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, matthews_corrcoef

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

# Create output folder for model artifacts if it doesn't exist
os.makedirs('model', exist_ok=True)

# Load dataset (Replace 'winequality-red.csv' with your dataset path)
df = pd.read_csv('winequality-red.csv', sep=';')

# Binary target conversion (e.g., quality >= 6 is 1, else 0)
df['target'] = (df['quality'] >= 6).astype(int)
X = df.drop(columns=['quality', 'target'])
y = df['target']

# Train-Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Save test data for Streamlit upload requirement
test_data = X_test.copy()
test_data['target'] = y_test
test_data.to_csv('test_data.csv', index=False)

# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
joblib.dump(scaler, 'scaler.joblib')

# Models definition
models = {
    'Logistic Regression': LogisticRegression(),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'KNN': KNeighborsClassifier(),
    'Naive Bayes': GaussianNB(),
    'Support Vector Machine': SVC(probability=True, random_state=42)
}

metrics_list = []

for name, model in models.items():
    if name in ['Logistic Regression', 'KNN', 'Support Vector Machine']:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]
        
    # Save model artifact
    joblib.dump(model, f'model/{name.replace(" ", "_").lower()}.joblib')
    
    # Calculate required metrics
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    
    metrics_list.append({
        'ML Model Name': name,
        'Accuracy': round(acc, 4),
        'AUC': round(auc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1': round(f1, 4),
        'MCC': round(mcc, 4)
    })

metrics_df = pd.DataFrame(metrics_list)
print(metrics_df)

/home/cloud/anaconda3/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


            ML Model Name  Accuracy     AUC  Precision  Recall      F1     MCC
0     Logistic Regression    0.7406  0.8242     0.7683  0.7368  0.7522  0.4808
1           Decision Tree    0.7531  0.7513     0.7644  0.7778  0.7710  0.5034
2                     KNN    0.7406  0.8117     0.7588  0.7544  0.7566  0.4790
3             Naive Bayes    0.7250  0.7909     0.7785  0.6784  0.7250  0.4569
4  Support Vector Machine    0.7625  0.8365     0.8146  0.7193  0.7640  0.5310
